In [1]:
# Cell 1: Install vLLM (Replaces Ollama)
!pip install vllm==0.6.4.post1 transformers==4.45.2 pandas tqdm


In [2]:
# Cell 2: Apply vLLM environment fixes
import os
import sys
import transformers

print(f"Transformers version: {transformers.__version__}")

os.environ["VLLM_USE_V1"] = "0"
os.environ["VLLM_CONFIGURE_LOGGING"] = "0"

if not hasattr(sys.stdout, 'fileno'):
    sys.stdout.fileno = lambda: 1
if not hasattr(sys.stderr, 'fileno'):
    sys.stderr.fileno = lambda: 2

print("✅ System fixes applied.")

In [3]:
# Cell 3: Setup and Drive Mount
from google.colab import drive
import pandas as pd
import json
import os
import time
from vllm import LLM, SamplingParams
from tqdm.auto import tqdm

drive.mount('/content/drive')

# --- CONFIGURATION ---
DATASET_PATH = "/content/drive/MyDrive/SFT/Datasets/beavertails_safe.csv" # Update to your safe dataset path
PROMPT_COLUMN = "prompt"
OUTPUT_FILE = "/content/drive/MyDrive/SFT/Processed datasets/beavertails_safe.csv_vllm.json"
MODEL_NAME = "TheBloke/Mistral-7B-Instruct-v0.2-AWQ"

# Optimized for T4 GPU (16GB VRAM)
llm = LLM(
    model=MODEL_NAME,
    quantization="awq",
    dtype="half",
    enforce_eager=True,
    gpu_memory_utilization=0.9,
    max_model_len=2048,
    max_num_seqs=32
)

# Sampling params for diverse generation options
sampling_params = SamplingParams(
    n=5,
    temperature=0.5,
    top_p=0.9,
    max_tokens=512 # Increased slightly for comprehensive safe responses
)

In [4]:
# Cell 4: Quality scoring logic for safe responses
def select_best_from_vllm(vllm_output):
    """
    Takes 5 parallel responses and selects the highest-quality,
    helpful response while avoiding accidental refusals.
    """
    # Signals that the model mistakenly refused a benign prompt
    false_refusal_words = ['cannot fulfill', 'unable to assist', 'as an ai, i refuse', 'violates policy', 'against my guidelines']

    candidates = [o.text.strip() for o in vllm_output.outputs]
    scored_attempts = []

    for resp in candidates:
        # Check if the model hallucinated a refusal
        has_false_refusal = any(word in resp.lower() for word in false_refusal_words)

        # Positive attributes
        sufficient_length = len(resp) > 100
        has_formatting = any(marker in resp for marker in ['\n', '*', '-', '1.'])

        # Calculate score (Start at 2, penalize heavily for false refusals)
        quality_score = 0
        if not has_false_refusal:
            quality_score += 2
        if sufficient_length:
            quality_score += 1
        if has_formatting:
            quality_score += 1

        scored_attempts.append({
            'response': resp,
            'score': quality_score,
            'length': len(resp)
        })

    # Pick the best score, use length as tie-breaker
    valid_attempts = [a for a in scored_attempts if a['score'] >= 3]
    if valid_attempts:
        best = max(valid_attempts, key=lambda x: x['length'])
    else:
        best = max(scored_attempts, key=lambda x: x['length'])

    return best['response'], best['score']

In [5]:
# Cell 5: Load Dataset + Resume Progress
import os
import json
import pandas as pd

df = pd.read_csv(DATASET_PATH)
all_prompts = df[PROMPT_COLUMN].tolist()
print(f"📊 Total safe prompts in dataset: {len(all_prompts)}")

# Load existing progress
if os.path.exists(OUTPUT_FILE):
    try:
        with open(OUTPUT_FILE, 'r', encoding='utf-8') as f:
            results = json.load(f)
        print(f"✅ Loaded {len(results)} existing results. Resuming...")
    except Exception as e:
        print(f"⚠️ Error loading file: {e}. Starting fresh.")
        results = {}
else:
    results = {}
    print("📝 Starting new results file.")

# Track processed items
processed_indices = set(results.keys())
remaining_prompts = [p for i, p in enumerate(all_prompts) if str(i) not in processed_indices]
remaining_indices = [i for i, p in enumerate(all_prompts) if str(i) not in processed_indices]

print(f"🚀 Remaining to process: {len(remaining_prompts)}")

In [7]:
# Cell 6: Chunked vLLM Processing (Helpful Responses for SFT)
import time
import json

CHUNK_SIZE = 20

if len(remaining_prompts) > 0:
    # Instruction to provide a stellar, normal assistant interaction
    system_instruction = """You are a highly helpful, accurate, and articulate AI assistant.
Your task is to completely and thoroughly answer the user's prompt.
Provide clear explanations, use markdown formatting (like bullet points or bold text) where appropriate to make your response structured and readable, and maintain a professional yet engaging tone."""

    print(f"🔥 Starting vLLM on {len(remaining_prompts)} prompts in chunks of {CHUNK_SIZE}...")
    start_time = time.time()

    for i in range(0, len(remaining_prompts), CHUNK_SIZE):
        chunk_prompts = remaining_prompts[i : i + CHUNK_SIZE]
        chunk_indices = remaining_indices[i : i + CHUNK_SIZE]

        # Mistral [INST] formatting
        formatted_chunk = [f"<s>[INST] {system_instruction}\n\n{p} [/INST]" for p in chunk_prompts]

        print(f"📦 Processing chunk {i//CHUNK_SIZE + 1}...")

        # Batch generation
        outputs = llm.generate(formatted_chunk, sampling_params)

        for j, output in enumerate(outputs):
            original_idx = str(chunk_indices[j])

            # Filter and extract using the helpful response scoring logic
            best_response, final_score = select_best_from_vllm(output)

            results[original_idx] = {
                "prompt": chunk_prompts[j],
                "helpful_response": best_response,
                "score": final_score,
                "timestamp": time.strftime("%Y-%m-%d %H:%M:%S"),
                "model": "mistral:7b-vllm-awq"
            }

        # Save progress checkpoint to Google Drive
        with open(OUTPUT_FILE, 'w', encoding='utf-8') as f:
            json.dump(results, f, indent=2, ensure_ascii=False)

        print(f"💾 Chunk saved. Total progress: {len(results)}/{len(all_prompts)}")

    print(f"🎉 All {len(remaining_prompts)} prompts processed successfully!")
    print(f"⏱️ Total Time: {(time.time()-start_time)/60:.1f} minutes.")
else:
    print("✅ All prompts already processed according to the output file.")